# ASL Sign Language Interpreter
## 3. Live-Inferenz (Webcam)
Lädt das trainierte Modell und klassifiziert ASL-Zeichen in Echtzeit.

> **Voraussetzung:** `02_training.ipynb` wurde ausgeführt und `model_resnet18_asl_best.pth` existiert.

### Imports & Konfiguration

In [1]:
import os
import torch
import cv2
from torch import nn
from torchvision import datasets, models, transforms

DATA_DIR   = "../data/offline_composited"
MODEL_PATH = "../model_resnet18_asl_best.pth"
DEVICE     = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

NUM_CLASSES          = 29
CONFIDENCE_THRESHOLD = 0.6
PADDING              = 20  # Pixel Abstand um die Hand herum

print(f"Device: {DEVICE}")

Device: mps


### Modell laden

In [2]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(in_features=512, out_features=NUM_CLASSES)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# Klassennamen aus dem Trainings-Datensatz lesen
class_names = datasets.ImageFolder(root=DATA_DIR).classes
print(f"Modell geladen! Klassen: {class_names}")

Modell geladen! Klassen: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


### Live-Inferenz
Drücke **Q** zum Beenden.

In [3]:
transform_live = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

cap = cv2.VideoCapture(0)

BOX_SIZE = 300  # Größe der festen ROI-Box in Pixeln

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w = frame.shape[:2]

    # Feste Quadrat-Box (mittig platziert)
    x1 = (w - BOX_SIZE) // 2
    x2 = x1 + BOX_SIZE
    y1 = (h - BOX_SIZE) // 2
    y2 = y1 + BOX_SIZE

    # Sicherstellen, dass die Koordinaten nicht aus dem Bild ragen
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)

    roi = frame[y1:y2, x1:x2]

    # Preprocessing für das PyTorch-Modell
    rgb_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)

    # WICHTIG: Der Frame wurde oben für die Spiegel-Ansicht geflippt (cv2.flip(frame, 1)).
    # Dadurch sieht eine rechte Hand für das Modell aus wie eine linke Hand!
    # Wir spiegeln den Ausschnitt für das Modell wieder zurück in die Originalperspektive.
    rgb_roi_model = cv2.flip(rgb_roi, 1)

    img_tensor = transform_live(rgb_roi_model).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probs = torch.softmax(model(img_tensor), dim=1)
        confidence, predicted = torch.max(probs, 1)

    conf_val   = confidence.item()
    pred_class = class_names[predicted.item()]

    if conf_val >= CONFIDENCE_THRESHOLD:
        label = f"{pred_class}  {conf_val*100:.1f}%"
        color = (255, 0, 0)
    else:
        label = f"?  {conf_val*100:.1f}%"
        color = (0, 0, 255)

    # Das feste Quadrat anzeigen
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    # Label direkt ÜBER der Box platzieren
    cv2.putText(frame, label, (x1, y1 - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 2)

    cv2.imshow("ASL Interpreter", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 